# Recovery Risk Calibration — Research Audit

## Research question

How sensitive are portfolio credit-loss estimates to the assumed recovery-rate distribution when expected recovery is held constant?

## Models
- Parametric Beta distribution
- Boundary-corrected reflected Kernel Density Estimation (KDE)

## Method
Both fitted distributions are calibrated to the same empirical mean recovery. Portfolio loss metrics are then compared, with repeated experiments used to examine Monte Carlo variability.

**Data status:** All recovery observations are synthetic. Results are illustrative and are not real-world credit-risk estimates.

In [3]:
from pathlib import Path
import sys
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import beta

# Find the project root from the current notebook location
current = Path.cwd().resolve()

candidates = [current, *current.parents]
ROOT = next(
    (path for path in candidates if (path / "main.py").exists()
     and (path / "src").is_dir()),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "Could not find the project root containing main.py and src/. "
        f"Notebook working directory: {current}"
    )

# Add the project root so Python can import src and main
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)

from src.data_simulator import generate_synthetic_recoveries
from src.density_estimators import (
    fit_parametric_beta,
    fit_reflection_kde,
    bounded_kde_density,
)

from main import (
    normalize_density,
    match_density_mean,
    density_mean,
)

REPORTS = ROOT / "reports"

print("Project root:", ROOT)
print("src folder exists:", (ROOT / "src").is_dir())
print("Reports folder exists:", REPORTS.is_dir())
print("Python interpreter:", sys.executable)

Project root: C:\Users\swaya\Documents\recovery-risk-calibration
src folder exists: True
Reports folder exists: True
Python interpreter: c:\Users\swaya\Documents\recovery-risk-calibration\.venv\Scripts\python.exe


In [4]:
N_OBSERVATIONS = 5_000
SEED = 2026
GRID_SIZE = 20_001

recoveries = generate_synthetic_recoveries(
    n=N_OBSERVATIONS,
    scenario="bimodal",
    seed=SEED,
)

grid = np.linspace(0.0, 1.0, GRID_SIZE)

target_mean = float(np.mean(recoveries))

print(f"Observations: {len(recoveries):,}")
print(f"Empirical mean recovery: {target_mean:.6f}")
print(f"Empirical standard deviation: {np.std(recoveries, ddof=1):.6f}")
print(f"Minimum recovery: {recoveries.min():.6f}")
print(f"Maximum recovery: {recoveries.max():.6f}")

Observations: 5,000
Empirical mean recovery: 0.496011
Empirical standard deviation: 0.307334
Minimum recovery: 0.006535
Maximum recovery: 0.997357


In [ ]:
alpha, beta_param = fit_parametric_beta(recoveries)
kde = fit_reflection_kde(recoveries)

beta_density = beta.pdf(
    grid,
    a=alpha,
    b=beta_param,
    loc=0,
    scale=1,
)

beta_density = normalize_density(grid, beta_density)

kde_density = bounded_kde_density(kde, grid)

beta_density, beta_tilt = match_density_mean(
    grid, beta_density, target_mean
)

kde_density, kde_tilt = match_density_mean(
    grid, kde_density, target_mean
)

beta_mean = density_mean(grid, beta_density)
kde_mean = density_mean(grid, kde_density)

calibration_summary = pd.DataFrame({
    "Model": ["Beta", "Reflected KDE"],
    "Calibrated mean": [beta_mean, kde_mean],
    "Target mean": [target_mean, target_mean],
    "Absolute mean error": [
        abs(beta_mean - target_mean),
        abs(kde_mean - target_mean),
    ],
    "Exponential tilt": [beta_tilt, kde_tilt],
})

display(calibration_summary)

print(f"Beta parameters: alpha={alpha:.6f}, beta={beta_param:.6f}")

ValueError: Grid must be strictly increasing.

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    recoveries,
    bins=50,
    density=True,
    alpha=0.25,
    color="grey",
    label="Synthetic observations",
)

plt.plot(
    grid, beta_density,
    linewidth=2,
    label="Mean-matched Beta",
)

plt.plot(
    grid, kde_density,
    linewidth=2,
    label="Mean-matched reflected KDE",
)

plt.axvline(
    target_mean,
    linestyle="--",
    linewidth=1,
    label=f"Target mean = {target_mean:.3f}",
)

plt.xlabel("Recovery rate")
plt.ylabel("Density")
plt.title("Recovery distributions after mean calibration")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
primary_results = pd.DataFrame({
    "Risk metric": [
        "Expected loss",
        "VaR 95%",
        "ES 95%",
        "Economic capital 95%",
        "VaR 99%",
        "ES 99%",
        "Economic capital 99%",
    ],
    "Beta model ($)": [
        249925.29,
        478593.16,
        554422.25,
        228667.87,
        595659.36,
        659112.57,
        345734.07,
    ],
    "Reflected KDE ($)": [
        250257.71,
        486460.00,
        559092.76,
        236202.29,
        603072.14,
        667314.95,
        352814.43,
    ],
})

primary_results["KDE minus Beta ($)"] = (
    primary_results["Reflected KDE ($)"]
    - primary_results["Beta model ($)"]
)

display(primary_results.style.format({
    "Beta model ($)": "${:,.2f}",
    "Reflected KDE ($)": "${:,.2f}",
    "KDE minus Beta ($)": "${:,.2f}",
}))

In [ ]:
repeated_path = REPORTS / "repeated_experiments.csv"
summary_path = REPORTS / "repeated_experiment_summary.csv"

repeated = pd.read_csv(repeated_path)
repeated_summary = pd.read_csv(summary_path)

print("Repeated experiment rows:", len(repeated))
display(repeated.head())
display(repeated_summary)

In [ ]:
gap_columns = {
    "VaR 99%": "VaR 99%",
    "ES 99%": "ES 99%",
    "Economic capital 99%": "Economic capital 99%",
}

plt.figure(figsize=(10, 5))

for label, column in gap_columns.items():
    if column in repeated.columns:
        plt.plot(
            repeated["experiment"],
            repeated[column],
            marker="o",
            markersize=3,
            label=label,
        )

plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.xlabel("Experiment")
plt.ylabel("KDE minus Beta ($)")
plt.title("Repeated-experiment risk gaps")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Findings

The primary experiment produced a positive KDE-minus-Beta gap for all seven reported risk metrics. The mean recovery rate was matched across the two fitted distributions, but their distributional shapes differed.

Across 30 repeated Monte Carlo experiments:
- The mean 99% VaR gap was approximately $9,901.
- The mean 99% expected-shortfall gap was approximately $12,418.
- The mean 99% economic-capital gap was approximately $9,776.
- The KDE economic-capital estimate exceeded the Beta estimate in 28 of 30 experiments.

## Interpretation

These findings illustrate that matching expected recovery alone does not ensure matching portfolio tail-risk estimates. The fitted recovery distribution can matter when portfolio losses are simulated.

## Limitations

- The recovery sample is synthetic.
- The fitted distributions are held fixed across repeated experiments.
- The repeated runs measure Monte Carlo variability, not uncertainty from refitting the distributions.
- The model assumes homogeneous default probabilities and fixed exposures.
- Default dependence and dependence between recovery and default conditions are not modelled.
- The results are specific to this synthetic scenario and configuration.

The results do not establish that KDE is universally more conservative or more accurate than the Beta model.